# Student Feedback Sentiment Analysis
**Pipeline**: Web Scraping → Preprocessing (NLTK) → ML Models → Visualization → Power BI Export

In [ ]:
import pandas as pd
import numpy as np
import re
import time
import warnings
from collections import Counter
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from textblob import TextBlob
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_curve, auc

warnings.filterwarnings('ignore')
for pkg in ['punkt', 'punkt_tab', 'stopwords']:
    nltk.download(pkg, quiet=True)

## 1. Data Collection
Scrape Google Maps reviews for a college using Selenium, or load from a CSV file.

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

def scrape_google_reviews(maps_url, max_reviews=300):
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    options.add_experimental_option('prefs', {'intl.accept_languages': 'en,en_US'})

    driver = webdriver.Chrome(options=options)
    reviews_data = []
    seen_ids = set()

    try:
        driver.get(maps_url)
        wait = WebDriverWait(driver, 15)
        time.sleep(3)

        # Click Reviews tab
        try:
            tab = wait.until(EC.element_to_be_clickable(
                (By.XPATH, '//button[contains(@aria-label, "Reviews")]')
            ))
            tab.click()
            time.sleep(2)
        except Exception as e:
            print(f'Reviews tab: {e}')

        # Sort by newest for temporal analysis
        try:
            sort_btn = wait.until(EC.element_to_be_clickable(
                (By.XPATH, '//button[@aria-label="Sort reviews"]')
            ))
            sort_btn.click()
            time.sleep(1)
            newest = wait.until(EC.element_to_be_clickable(
                (By.XPATH, '//li[@aria-label="Newest"]')
            ))
            newest.click()
            time.sleep(2)
        except Exception as e:
            print(f'Sort: {e}')

        # Scrollable container
        try:
            scrollable = driver.find_element(
                By.XPATH, '//div[@role="main"]//div[contains(@class,"m6QErb")][@tabindex="-1"]'
            )
        except:
            scrollable = driver.find_element(By.TAG_NAME, 'body')

        prev_count = 0
        stall = 0

        while len(reviews_data) < max_reviews:
            # Expand truncated reviews
            for btn in driver.find_elements(By.XPATH, '//button[contains(@aria-label,"See more")]'):
                try:
                    driver.execute_script('arguments[0].click();', btn)
                    time.sleep(0.2)
                except:
                    pass

            for block in driver.find_elements(By.XPATH, '//div[@data-review-id]'):
                rid = block.get_attribute('data-review-id')
                if rid in seen_ids:
                    continue
                try:
                    aria = block.find_element(
                        By.XPATH, './/span[contains(@aria-label,"star")]'
                    ).get_attribute('aria-label')
                    rating = int(re.search(r'\d+', aria).group())
                except:
                    rating = None
                try:
                    text = block.find_element(
                        By.XPATH, './/span[contains(@class,"wiI7pd")]'
                    ).text.strip()
                except:
                    text = ''
                try:
                    date = block.find_element(
                        By.XPATH, './/span[contains(@class,"rsqaWe")]'
                    ).text.strip()
                except:
                    date = ''
                try:
                    author = block.find_element(
                        By.XPATH, './/div[contains(@class,"d4r55")]'
                    ).text.strip()
                except:
                    author = ''
                if text:
                    seen_ids.add(rid)
                    reviews_data.append({'author': author, 'rating': rating,
                                         'text': text, 'date': date})

            print(f'Collected {len(reviews_data)} reviews...')
            driver.execute_script('arguments[0].scrollTop += 3000', scrollable)
            time.sleep(2)

            if len(reviews_data) == prev_count:
                stall += 1
                if stall >= 3:
                    print('No new reviews, stopping.')
                    break
            else:
                stall = 0
                prev_count = len(reviews_data)

    finally:
        driver.quit()

    return pd.DataFrame(reviews_data)

## 2. Load Data
Uncomment **Option 1** to scrape live data, or use **Option 2** to load from a CSV.

In [ ]:
# --- OPTION 1: Scrape Google Maps ---
# MAPS_URL = 'https://www.google.com/maps/place/Your+College+Name'
# df = scrape_google_reviews(MAPS_URL, max_reviews=300)
# df.to_csv('reviews_raw.csv', index=False)

# --- OPTION 2: Load from CSV ---
# Required column: 'text'   Optional column: 'rating' (1-5 stars)
CSV_PATH = 'reviews_raw.csv'
df = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df)} reviews')
df.head()

## 3. Data Exploration

In [ ]:
print('Shape:', df.shape)
print('\nNull counts:')
print(df.isnull().sum())

if 'rating' in df.columns:
    print('\nRating distribution:')
    print(df['rating'].value_counts().sort_index())
    df['rating'].value_counts().sort_index().plot(
        kind='bar', color='steelblue', figsize=(6, 4), edgecolor='black'
    )
    plt.title('Rating Distribution')
    plt.xlabel('Stars')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 4. Text Preprocessing
Tokenization, stopword removal, and stemming using NLTK.

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)      # remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)          # keep only letters
    tokens = word_tokenize(text)                      # NLTK tokenize
    tokens = [stemmer.stem(w) for w in tokens        # stem + remove stopwords
              if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

df['cleaned_text'] = df['text'].apply(preprocess_text)
df[['text', 'cleaned_text']].head()

## 5. Sentiment Labeling
Uses star ratings as ground truth (1-2=negative, 3=neutral, 4-5=positive). Falls back to TextBlob if no rating column.

In [ ]:
def label_sentiment(row):
    if 'rating' in df.columns and pd.notna(row.get('rating')):
        r = int(row['rating'])
        if r >= 4:
            return 'positive'
        elif r == 3:
            return 'neutral'
        else:
            return 'negative'
    polarity = TextBlob(str(row['text'])).sentiment.polarity
    if polarity > 0:
        return 'positive'
    elif polarity < 0:
        return 'negative'
    return 'neutral'

df['sentiment'] = df.apply(label_sentiment, axis=1)
print(df['sentiment'].value_counts())

## 6. Feature Extraction (TF-IDF)

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 3))
X = vectorizer.fit_transform(df['cleaned_text'])
y = df['sentiment']
print('Feature matrix shape:', X.shape)

## 7. Train / Test Split

In [ ]:
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
except ValueError:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
print(f'Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}')

## 8. Model Training
Logistic Regression, Naive Bayes, SVM.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Naive Bayes':         MultinomialNB(),
    'SVM':                 SVC(kernel='linear', probability=True)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[name] = {'model': model, 'preds': preds, 'accuracy': acc}
    print(f'\n{name} — Accuracy: {acc:.4f}')
    print(classification_report(y_test, preds))

## 9. Hyperparameter Tuning (Logistic Regression)

In [ ]:
param_grid = {
    'C':      [0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'saga']
}
grid = GridSearchCV(
    LogisticRegression(max_iter=1000), param_grid,
    cv=5, scoring='accuracy', n_jobs=-1
)
grid.fit(X_train, y_train)

print('Best params:', grid.best_params_)
print('Best CV score: {:.4f}'.format(grid.best_score_))

best_model = grid.best_estimator_
best_preds = best_model.predict(X_test)
print('Test accuracy: {:.4f}'.format(accuracy_score(y_test, best_preds)))

## 10. Confusion Matrix

In [ ]:
classes = ['negative', 'neutral', 'positive']
cm = confusion_matrix(y_test, best_preds, labels=classes)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix — Best Logistic Regression')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 11. ROC Curve (Multi-class, One-vs-Rest)

In [ ]:
y_bin = label_binarize(y, classes=['negative', 'neutral', 'positive'])
X_tr, X_te, y_tr_bin, y_te_bin = train_test_split(
    X, y_bin, test_size=0.2, random_state=42
)

clf = OneVsRestClassifier(LogisticRegression(max_iter=1000))
clf.fit(X_tr, y_tr_bin)
y_score = clf.decision_function(X_te)

plt.figure(figsize=(9, 6))
for i, (cls, color) in enumerate(zip(
    ['negative', 'neutral', 'positive'], ['blue', 'orange', 'green']
)):
    fpr, tpr, _ = roc_curve(y_te_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{cls} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 12. Visualizations

In [ ]:
sentiment_colors = {'positive': '#99ff99', 'negative': '#ff9999', 'neutral': '#66b3ff'}
counts = df['sentiment'].value_counts()
bar_colors = [sentiment_colors.get(s, 'gray') for s in counts.index]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

counts.plot(kind='bar', ax=axes[0], color=bar_colors, edgecolor='black')
axes[0].set_title('Sentiment Distribution')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

axes[1].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=bar_colors, startangle=140,
            explode=[0.05] * len(counts))
axes[1].set_title('Sentiment Share')

plt.tight_layout()
plt.show()

In [ ]:
def parse_date(date_str):
    date_str = str(date_str).lower()
    now = datetime.now()
    m = re.search(r'(\d+)', date_str)
    n = int(m.group()) if m else 1
    if 'year'  in date_str: return now - timedelta(days=n * 365)
    if 'month' in date_str: return now - timedelta(days=n * 30)
    if 'week'  in date_str: return now - timedelta(weeks=n)
    if 'day'   in date_str: return now - timedelta(days=n)
    return now

if 'date' in df.columns:
    df['parsed_date'] = df['date'].apply(parse_date)
    df['month'] = df['parsed_date'].dt.to_period('M')
    trend = df.groupby(['month', 'sentiment']).size().unstack(fill_value=0)
    trend.plot(figsize=(12, 5), marker='o')
    plt.title('Sentiment Trend Over Time')
    plt.xlabel('Month')
    plt.ylabel('Number of Reviews')
    plt.tight_layout()
    plt.show()
else:
    print('No date column — skipping trend chart')

## 13. Export for Power BI
Load `sentiment_for_powerbi.csv` into Power BI Desktop to build dashboards (sentiment trend, rating distribution, complaint areas).

In [ ]:
export_cols = [c for c in
    ['author', 'rating', 'text', 'sentiment', 'date', 'parsed_date']
    if c in df.columns
]
df[export_cols].to_csv('sentiment_for_powerbi.csv', index=False)
print(f'Exported {len(df)} rows to sentiment_for_powerbi.csv')
df[export_cols].head()

## 14. Improvement Suggestions
Extract top complaint keywords from negative reviews and map to actionable areas.

In [ ]:
negative_texts = df[df['sentiment'] == 'negative']['cleaned_text']

all_words = []
for text in negative_texts:
    all_words.extend(text.split())

word_freq = Counter(all_words)

categories = {
    'Teaching Quality': ['teacher', 'professor', 'faculti', 'lectur', 'teach', 'explain', 'cours'],
    'Infrastructure':   ['build', 'room', 'classroom', 'lab', 'librari', 'toilet', 'hostel', 'park'],
    'Canteen / Food':   ['food', 'canteen', 'cafeteria', 'eat', 'mess', 'lunch'],
    'Fees / Cost':      ['fee', 'expens', 'cost', 'money', 'payment', 'charg'],
    'Management':       ['manag', 'administr', 'principal', 'rule', 'disciplin', 'staff'],
    'Placement':        ['placement', 'job', 'campus', 'recruit', 'compani', 'hire']
}

issue_scores = {cat: 0 for cat in categories}
for word, count in word_freq.items():
    for cat, keywords in categories.items():
        if any(kw in word for kw in keywords):
            issue_scores[cat] += count

issue_scores = {k: v for k, v in
                sorted(issue_scores.items(), key=lambda x: x[1], reverse=True)
                if v > 0}

print('Top complaint areas from negative reviews:')
print('-' * 42)
for area, score in issue_scores.items():
    print(f'  {area:<22}  mention score: {score}')

if issue_scores:
    plt.figure(figsize=(8, 4))
    plt.bar(issue_scores.keys(), issue_scores.values(),
            color='salmon', edgecolor='black')
    plt.title('Areas Needing Improvement (from Negative Reviews)')
    plt.ylabel('Mention Score')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.show()

suggestions = {
    'Teaching Quality': 'Run faculty training programs; collect regular feedback on teaching methods.',
    'Infrastructure':   'Upgrade classrooms, labs, and common facilities.',
    'Canteen / Food':   'Improve food quality and variety in the canteen.',
    'Fees / Cost':      'Offer flexible payment plans or additional scholarship options.',
    'Management':       'Improve communication between administration and students.',
    'Placement':        'Strengthen industry tie-ups and pre-placement training programs.'
}

print('\nTop 3 Improvement Suggestions:')
for area in list(issue_scores.keys())[:3]:
    print(f'\n  [{area}]')
    print(f'  {suggestions.get(area, "Review student complaints in this area.")}'  )